# Content research crew: grounded, cited drafting

A content team that doesn't hallucinate: a **researcher** agent gathers real
sources from an [Infino](https://pypi.org/project/infino/) store, and a
**writer** agent drafts a short explainer that cites them by title — writing
only from what was retrieved. With
[`crewai-infino`](https://pypi.org/project/crewai-infino/) the researcher gets
semantic, BM25, hybrid, and SQL search as tools over **one** store, one copy of
the data — so every claim traces back to a document, with no second vector
database to keep in sync.

**The crew needs an LLM key** (the agents are tool-calling) — building and
searching the store is key-free. See the README.

In [1]:
import shutil
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # examples/ root, where _shared/ lives

import infino
import pyarrow as pa
from crewai import Agent, Crew, Process, Task
from crewai_infino import InfinoIndex

from _shared.crew import crew_llm
from _shared.embedding import DIM, METRIC, embed, embed_query
from _shared.loaders import load_arxiv
from _shared.sql import query

DB_DIR = "./content_data"
TABLE = "sources"
shutil.rmtree(DB_DIR, ignore_errors=True)  # start fresh each run

db = infino.connect(DB_DIR)
llm = crew_llm()
print("LLM:", "on" if llm else "off (build + search still run; crew is skipped)")

LLM: on


## Build the source library

One `InfinoIndex` over arXiv ML abstracts. Each abstract is short and
self-contained, so it's one row (no chunking); the paper `title` is a scalar
column the writer can cite. `add_texts` embeds and appends in one call.

In [2]:
papers = load_arxiv(n=300)
index = InfinoIndex.create(
    db, TABLE,
    embed_documents=embed, embed_query=embed_query, dim=DIM, metric=METRIC,
    metadata_columns=[pa.field("title", pa.large_utf8(), nullable=False)],
)
index.add_texts(
    [p["abstract"] for p in papers],
    metadatas=[{"title": p["title"]} for p in papers],
)
print("indexed", query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0], "abstracts")

indexed 300 abstracts


## Hybrid retrieval surfaces the sources (key-free)

Before any LLM, hybrid search (BM25 + vector, fused by RRF in one call) over the
store returns the documents a writer would cite — these titles are what the
draft must ground itself in.

In [3]:
TOPIC = "reinforcement learning"

for r in index.searcher.hybrid(TOPIC, k=3):
    print("-", (r.get("title") or "?")[:70])

- Rollout Sampling Approximate Policy Iteration
- Feature Reinforcement Learning: Part I: Unstructured MDPs
- Optimistic Simulated Exploration as an Incentive for Real Exploration


## The content crew

The **researcher** gathers and summarizes sources with the Infino tools; the
**writer** drafts a short explainer that cites them by title, using only the
researched findings. `index.as_tools()` hands the researcher all four search
tools at once.

In [4]:
def write_explainer(topic: str) -> None:
    if llm is None:
        print(f"topic: {topic}\n(needs an LLM key — skipped)")
        return
    researcher = Agent(
        role="Research Analyst",
        goal="Find the most relevant sources on a topic and pull out the key points.",
        backstory="You search the source library and ground every point in a real "
                  "document, never your own memory.",
        tools=index.as_tools(), llm=llm, verbose=False,
    )
    writer = Agent(
        role="Technical Writer",
        goal="Write a short, accurate explainer that cites its sources.",
        backstory="You write only from the researched findings and cite each source "
                  "by its title.",
        llm=llm, verbose=False,
    )
    research = Task(
        description=f"Topic: {topic}\nFind the most relevant sources and summarize the "
                    "key points, noting each source title.",
        expected_output="A short list of key points, each tied to a source title.",
        agent=researcher,
    )
    draft = Task(
        description="Write a ~150-word explainer on the topic for a technical blog. "
                    "Cite sources inline by title; use only the researched findings.",
        expected_output="A ~150-word explainer with inline source-title citations.",
        agent=writer, context=[research],
    )
    crew = Crew(agents=[researcher, writer], tasks=[research, draft],
                process=Process.sequential, verbose=False)
    # Notebooks run inside an event loop; run the sync crew in a worker thread.
    with ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(crew.kickoff).result()
    print(f"topic: {topic}\n\n{result}")


write_explainer(TOPIC)

topic: reinforcement learning

Reinforcement learning (RL) studies how an agent improves behavior through sequences of observations, actions, and rewards, but a core practical difficulty is turning messy, history-dependent experience into a usable state representation. Both *Feature Reinforcement Learning: Part I: Unstructured MDPs* and *Feature Markov Decision Processes* argue that choosing the right features is essential because standard RL methods rely on an approximately Markov state. Once such a representation is available, temporal-difference methods estimate long-term value efficiently; *Temporal Difference Updating without a Learning Rate* shows that TD-style discounted value updates can even be derived without a fixed learning-rate parameter. RL also depends on balancing exploration and exploitation: *The many faces of optimism - Extended version* highlights optimism under uncertainty as a key exploration principle, while *Optimistic Simulated Exploration as an Incentive for R

## Takeaway

One Infino store gave the crew semantic, keyword, hybrid, and SQL search as
agent tools over one copy of the data. The researcher grounded every point in a
real document; the writer cited them by title — no hallucinated sources, no
separate vector database.